In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import balanced_accuracy_score
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv("/kaggle/input/binary-buzz-cyber-labs-ml-25/train.csv")
test_df = pd.read_csv("/kaggle/input/binary-buzz-cyber-labs-ml-25/test.csv")

print(f"Train: {train_df.shape}, Test: {test_df.shape}")


all_cols = train_df.columns.tolist()
print(f"All columns: {all_cols[:10]}...")  

features = None
if any(col.startswith('F_') for col in all_cols):
    features = [col for col in all_cols if col.startswith('F_')]
elif len(all_cols) > 2:  
    features = [col for col in all_cols if col not in ['id', 'target']]
else:
   
    features = all_cols[1:-1] if len(all_cols) > 2 else all_cols[:-1]

print(f"✅ Found {len(features)} features: {features[:5]}...")

train_df[features] = train_df[features].replace([np.inf, -np.inf], np.nan)
test_df[features] = test_df[features].replace([np.inf, -np.inf], np.nan)
train_df[features] = train_df[features].fillna(train_df[features].median())
test_df[features] = test_df[features].fillna(test_df[features].median())

X = train_df[features].values.astype(np.float32)
y = train_df['target'].values.astype(np.int32)
X_test = test_df[features].values.astype(np.float32)
test_ids = test_df['id'].values

print(f"X shape: {X.shape}, y shape: {y.shape}")
assert X.shape[1] > 0, "No features found!"

def robust_scale(X):
    X_scaled = np.copy(X).astype(np.float32)
    for i in range(X.shape[1]):
        col = X_scaled[:, i]
        median = np.median(col)
        q75, q25 = np.percentile(col, [75, 25])
        iqr = q75 - q25
        if iqr > 0:
            upper, lower = median + 3*iqr, median - 3*iqr
            X_scaled[:, i] = np.clip(col, lower, upper)
            X_scaled[:, i] = (X_scaled[:, i] - median) / iqr
    return X_scaled

X_scaled = robust_scale(X)
X_test_scaled = robust_scale(X_test)


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_oof = np.zeros(len(X_scaled))
xgb_oof = np.zeros(len(X_scaled))
rf_test = np.zeros(len(X_test_scaled))
xgb_test = np.zeros(len(X_test_scaled))



for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
    X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    n_features = X_tr.shape[1]
    rf = RandomForestClassifier(
        n_estimators=1000, 
        max_depth=10, 
        min_samples_split=8, 
        min_samples_leaf=4,
        max_features=min(0.3, n_features), 
        class_weight='balanced',
        random_state=42, 
        n_jobs=-1
    )
    rf.fit(X_tr, y_tr)
    rf_oof[val_idx] = rf.predict_proba(X_val)[:, 1]
    rf_test += rf.predict_proba(X_test_scaled)[:, 1] / 5
    
    scale_pos = np.sum(y_tr == 0) / np.sum(y_tr == 1) if np.sum(y_tr == 1) > 0 else 1
    xgb = XGBClassifier(
        n_estimators=1000, max_depth=6, learning_rate=0.1, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=2.0,
        scale_pos_weight=scale_pos, random_state=42, n_jobs=-1
    )
    xgb.fit(X_tr, y_tr)
    xgb_oof[val_idx] = xgb.predict_proba(X_val)[:, 1]
    xgb_test += xgb.predict_proba(X_test_scaled)[:, 1] / 5
    
    print(f"Fold {fold+1}/5 ✓")

def best_threshold(probs, y_true):
    thresholds = np.arange(0.1, 0.9, 0.01)
    best_score, best_t = 0, 0.5
    for t in thresholds:
        preds = (probs >= t).astype(int)
        score = balanced_accuracy_score(y_true, preds)
        if score > best_score:
            best_score, best_t = score, t
    return best_t

rf_thresh = best_threshold(rf_oof, y)
xgb_thresh = best_threshold(xgb_oof, y)
final_thresh = (rf_thresh + xgb_thresh) / 2

ensemble_proba = 0.6 * xgb_test + 0.4 * rf_test
y_pred = (ensemble_proba >= final_thresh).astype(int)

submission = pd.DataFrame({'id': test_ids, 'target': y_pred})
submission.to_csv('submission.csv', index=False)

print("\ SUBMISSION.csv SAVED!")
print(f"CV Balanced Acc: RF={balanced_accuracy_score(y, rf_oof>=rf_thresh):.4f}")
print(f"XGB={balanced_accuracy_score(y, xgb_oof>=xgb_thresh):.4f}")
print(f"Threshold: {final_thresh:.3f}")
print(f"Preds: {np.bincount(y_pred)}")


Train: (2000, 3074), Test: (2000, 3073)
All columns: ['id', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9']...
✅ Found 3072 features: ['x1', 'x2', 'x3', 'x4', 'x5']...
X shape: (2000, 3072), y shape: (2000,)
🚀 Training 5-Fold Stratified CV...
Fold 1/5 ✓
Fold 2/5 ✓
Fold 3/5 ✓
Fold 4/5 ✓
Fold 5/5 ✓

🎯 SUBMISSION.csv SAVED!
CV Balanced Acc: RF=0.6903
XGB=0.6089
Threshold: 0.120
Preds: [1698  302]
